# Exercises — Chapter 14: Financial Text Summarization and Information Extraction

Complete the exercises from Lecture 14.

In [ ]:
# Your code here

## Data Lab -- SEC EDGAR

Process 10-K filing text end-to-end: section extraction, extractive summarisation, and heuristic information extraction.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course your-email@example.com"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

### Exercise [B]: MD&A Section Extraction

In [ ]:
# --- Exercise [B]: MD&A Extraction ---
html_aapl = fetch_10k_html("AAPL")
mda_aapl  = extract_mda(html_aapl)

sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", mda_aapl) if len(s.split()) >= 5]
words     = mda_aapl.split()
print(f"Word count:     {len(words)}")
print(f"Sentence count: {len(sentences)}")
print(f"\nFirst 500 characters:\n{mda_aapl[:500]}...")

sent_lens = [len(s.split()) for s in sentences]
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(sent_lens, bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("Words per sentence"); ax.set_ylabel("Count")
ax.set_title("AAPL 10-K MD&A -- Sentence Length Distribution")
plt.tight_layout(); plt.show()

### Exercise [I]: Extractive Summarisation

In [ ]:
# --- Exercise [I]: TF-IDF Extractive Summarisation ---
def extractive_summary(text, top_n=5):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 5]
    if not sents: return [], 0.0
    def tok(s): return re.findall(r"[a-z]+", s.lower())
    all_toks = [tok(s) for s in sents]
    N = len(sents)
    df_cnt = {}
    for toks in all_toks:
        for w in set(toks): df_cnt[w] = df_cnt.get(w, 0) + 1
    idf_s = {w: np.log((N+1)/(c+1)) for w, c in df_cnt.items()}
    scores = []
    for toks in all_toks:
        n = max(len(toks), 1)
        tf = {}
        for w in toks: tf[w] = tf.get(w, 0) + 1/n
        sc = sum(tf[w] * idf_s.get(w, 0) for w in tf)
        scores.append(sc)
    top_idx = sorted(range(len(scores)), key=lambda i: -scores[i])[:top_n]
    summary_sents = [sents[i] for i in sorted(top_idx)]
    ratio = sum(len(s.split()) for s in summary_sents) / max(len(text.split()), 1)
    return summary_sents, ratio

for ticker in ["AAPL","MSFT","JPM"]:
    html = fetch_10k_html(ticker)
    mda  = extract_mda(html)
    sums, ratio = extractive_summary(mda)
    print(f"\n=== {ticker} | Compression ratio: {ratio:.3f} ===")
    for j, s in enumerate(sums, 1):
        print(f"[{j}] {s[:200]}")

### Exercise [A]: Information Extraction Pipeline

In [ ]:
# --- Exercise [A]: IE Pipeline ---
TICKERS_14 = ["AAPL","MSFT","GOOGL","AMZN","META"]

def extract_revenue(text):
    m = re.search(r"\$[\d,]+\.?\d*\s*(billion|million)", text, re.IGNORECASE)
    return m.group(0) if m else "Not found"

def extract_risk_sents(html, n=3):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(r"Item\s+1A[.\s]+Risk\s+Factors.*?(?=Item\s+1B|Item\s+2)", text, re.IGNORECASE | re.DOTALL)
    if not m: return []
    sec = re.sub(r"\s+", " ", m.group(0))
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", sec) if len(s.split()) >= 8]
    return sents[:n]

def extract_executives(html, n=3):
    text  = re.sub(r"<[^>]+>", " ", html)
    title_pat = r"(?:CEO|CFO|President|Chairman|Chief\s+\w+\s+Officer)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})"
    names = re.findall(title_pat, text)
    seen, unique = set(), []
    for name in names:
        if name not in seen: seen.add(name); unique.append(name)
        if len(unique) >= n: break
    return unique

results_14 = []
for t in TICKERS_14:
    html = fetch_10k_html(t)
    mda  = extract_mda(html)
    rec  = {
        "ticker":     t,
        "revenue":    extract_revenue(mda),
        "risk_sents": extract_risk_sents(html),
        "executives": extract_executives(html),
    }
    results_14.append(rec)
    print(f"\n=== {t} ===")
    print(f"Revenue:    {rec['revenue']}")
    print(f"Executives: {rec['executives']}")
    for i, rs in enumerate(rec["risk_sents"],1):
        print(f"Risk [{i}]: {rs[:150]}...")